# Campaña completa del paper: 20k Adam desde cero

Este notebook reproduce la campaña completa con una regla común: **20 000 épocas Adam, inicialización aleatoria, semilla 20260728, float64 y sin L-BFGS**. Esto mantiene comparable el presupuesto de optimización, incluso en el converso con gradient routing, donde L-BFGS no es válido.

Incluye baselines directo/converso, mixta vs three-field, arquitectura, SiLU vs tanh, routing on/off, sensibilidad de puntos interiores y barridos independientes de fuerza/voltaje. Cada corrida guarda modelo, tiempo, parámetros, loss total y componentes. Al final se calculan L2 relativo, RMSE, MAE, máximo absoluto y nRMSE sobre la misma malla FEM P2 de 201×21, y se genera un paquete con el estilo de `output/results_package/`, incluidas las comparaciones espaciales 2×3 con escalas de color compartidas.

La campaña contiene 17 entrenamientos y debe ejecutarse por grupos o uno por sesión. Las corridas completas se omiten automáticamente.

In [6]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
bundle = Path('/content/drive/MyDrive/pinn_piezo_colab_bundle.zip')
assert bundle.exists(), f'No se encontró {bundle}'
!rm -rf /content/PINNs_piezoelectricity
!mkdir -p /content/PINNs_piezoelectricity
!unzip -q "{bundle}" -d /content/PINNs_piezoelectricity
%cd /content/PINNs_piezoelectricity
%pip install -q --no-deps scikit-fem torchsummary pyDOE


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
/content/PINNs_piezoelectricity


In [7]:
import os, subprocess, sys, time
from pathlib import Path
import torch

PROJECT = Path('/content/PINNs_piezoelectricity')
SRC = PROJECT / 'src'
OUTPUTS = Path('/content/drive/MyDrive/PINNs_piezo_20k_outputs')
OUTPUTS.mkdir(parents=True, exist_ok=True)
os.environ['PINN_PIEZO_OUTPUTS_DIR'] = str(OUTPUTS)
os.environ['PYTHONPATH'] = str(SRC) + os.pathsep + str(PROJECT)
os.environ['PYTHONUNBUFFERED'] = '1'
sys.path[:0] = [str(SRC), str(PROJECT)]
assert torch.cuda.is_available(), 'Selecciona Runtime > Change runtime type > GPU.'
gpu = torch.cuda.get_device_name(0)
probe = torch.ones(8, device='cuda', dtype=torch.float64)
assert probe.dtype == torch.float64
print('GPU:', gpu, '| PyTorch:', torch.__version__, '| outputs:', OUTPUTS)
if 'T4' in gpu:
    print('Aviso: T4 ejecuta FP64, pero A100/V100 será mucho más rápida.')

def stream(command):
    command = list(map(str, command))
    child_env = os.environ.copy()
    print('Ejecutando:', ' '.join(command), flush=True)
    process = subprocess.Popen(
        command, cwd=PROJECT, env=child_env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

def campaign(names):
    stream([sys.executable, '-u', '-m', 'scripts.run_paper20k_campaign',
            '--epochs', '20000', '--device', 'cuda', '--dtype', 'float64',
            '--experiments', *names])


GPU: Tesla T4 | PyTorch: 2.11.0+cu128 | outputs: /content/drive/MyDrive/PINNs_piezo_20k_outputs
Aviso: T4 ejecuta FP64, pero A100/V100 será mucho más rápida.


## 1. Baselines mixtos

Ejecuta primero estas dos corridas. Son los modelos principales que alimentan las figuras FEM–PINN y las curvas de pérdidas por componente.

In [3]:
# campaign(['direct_mixed'])
# En otra sesión de Colab:
campaign(['indirect_mixed'])


Ejecutando: /usr/bin/python3 -u -m scripts.run_paper20k_campaign --epochs 20000 --device cuda --dtype float64 --experiments indirect_mixed

[START] indirect_mixed (baseline)
/usr/bin/python3 -u -m scripts.train_indirect --epochs-adam 20000 --device cuda --dtype float64 --epochs-lbfgs 0 --lr-adam 1e-3 --model-type pyramid --hidden-sizes 50 50 50 --activation silu --phase-enriched --no-split-trunks --no-hard-natural-bcs --interior-per-layer 2048 --boundary-points 256 --interface-points 256 --resample-every 200 --interface-weight 1.0 --no-slender-transverse-scaling --normal-constitutive-weight 1.0 --constitutive-normalization represented --no-constitutive-traction-bcs --no-constitutive-flux-stopgrad --mechanical-constitutive-stopgrad --stress-warmup-epochs 0 --seed 20260728 --voltage 100 --no-zero-output --no-periodic-checkpoints --run-name paper20k_indirect_mixed
Using device: cuda
Training dtype: torch.float64
Run directory: /content/drive/MyDrive/PINNs_piezo_20k_outputs/runs/paper20k_i

## 2. Mixta vs three-field

Los modelos three-field usan las mismas capas 3×50, SiLU, puntos, semilla y 20k Adam. Requieren segundas derivadas y pueden tardar considerablemente más.

In [ ]:
# campaign(['direct_three_field'])
# En otra sesión:
# campaign(['indirect_three_field'])


Ejecutando: /usr/bin/python3 -u -m scripts.run_paper20k_campaign --epochs 20000 --device cuda --dtype float64 --experiments direct_three_field
[SKIP] direct_three_field: completed model exists at /content/drive/MyDrive/PINNs_piezo_20k_outputs/runs/paper20k_direct_three_field/models/model_PINN_direct_three_field.pt


In [ ]:
# campaign(['indirect_three_field'])

Ejecutando: /usr/bin/python3 -u -m scripts.run_paper20k_campaign --epochs 20000 --device cuda --dtype float64 --experiments indirect_three_field

[START] indirect_three_field (mixed_vs_three_field)
/usr/bin/python3 -u -m scripts.train_indirect_three_field --epochs-adam 20000 --device cuda --dtype float64 --epochs-lbfgs 0 --lr-adam 1e-3 --activation silu --hidden-sizes 50 50 50 --phase-enriched --interface-enriched --interior-per-layer 2048 --boundary-points 256 --interface-points 256 --resample-every 200 --seed 20260728 --voltage 100 --run-name paper20k_indirect_three_field
Using device: cuda; threads: 1
Run directory: /content/drive/MyDrive/PINNs_piezo_20k_outputs/runs/paper20k_indirect_three_field
Initialization: cold random weights
[ADAM] 0/20000 loss=2.377686e+02
[ADAM] 100/20000 loss=6.360769e+00
[ADAM] 200/20000 loss=6.067130e+00
[ADAM] 300/20000 loss=5.744446e+00
[ADAM] 400/20000 loss=5.352681e+00
[ADAM] 500/20000 loss=4.768406e+00
[ADAM] 600/20000 loss=3.971114e+00
[ADAM] 700/2

KeyboardInterrupt: 

## 3. Arquitectura, activación y gradient routing (converso)

In [4]:
# Ejecuta uno por sesión o una lista corta si la GPU asignada es rápida.
campaign(['indirect_arch_2x50'])
# campaign(['indirect_arch_4x50'])
# campaign(['indirect_arch_3x100'])
# campaign(['indirect_tanh_3x50'])
# campaign(['indirect_routing_off'])


Ejecutando: /usr/bin/python3 -u -m scripts.run_paper20k_campaign --epochs 20000 --device cuda --dtype float64 --experiments indirect_arch_2x50

[START] indirect_arch_2x50 (architecture)
/usr/bin/python3 -u -m scripts.train_indirect --epochs-adam 20000 --device cuda --dtype float64 --epochs-lbfgs 0 --lr-adam 1e-3 --model-type pyramid --hidden-sizes 50 50 --activation silu --phase-enriched --no-split-trunks --no-hard-natural-bcs --interior-per-layer 2048 --boundary-points 256 --interface-points 256 --resample-every 200 --interface-weight 1.0 --no-slender-transverse-scaling --normal-constitutive-weight 1.0 --constitutive-normalization represented --no-constitutive-traction-bcs --no-constitutive-flux-stopgrad --mechanical-constitutive-stopgrad --stress-warmup-epochs 0 --seed 20260728 --voltage 100 --no-zero-output --no-periodic-checkpoints --run-name paper20k_indirect_arch_2x50
Using device: cuda
Training dtype: torch.float64
Run directory: /content/drive/MyDrive/PINNs_piezo_20k_outputs/ru

In [5]:
campaign(['indirect_arch_4x50'])
# campaign(['indirect_arch_3x100'])
# campaign(['indirect_tanh_3x50'])
# campaign(['indirect_routing_off'])

Ejecutando: /usr/bin/python3 -u -m scripts.run_paper20k_campaign --epochs 20000 --device cuda --dtype float64 --experiments indirect_arch_4x50

[START] indirect_arch_4x50 (architecture)
/usr/bin/python3 -u -m scripts.train_indirect --epochs-adam 20000 --device cuda --dtype float64 --epochs-lbfgs 0 --lr-adam 1e-3 --model-type pyramid --hidden-sizes 50 50 50 50 --activation silu --phase-enriched --no-split-trunks --no-hard-natural-bcs --interior-per-layer 2048 --boundary-points 256 --interface-points 256 --resample-every 200 --interface-weight 1.0 --no-slender-transverse-scaling --normal-constitutive-weight 1.0 --constitutive-normalization represented --no-constitutive-traction-bcs --no-constitutive-flux-stopgrad --mechanical-constitutive-stopgrad --stress-warmup-epochs 0 --seed 20260728 --voltage 100 --no-zero-output --no-periodic-checkpoints --run-name paper20k_indirect_arch_4x50
Using device: cuda
Training dtype: torch.float64
Run directory: /content/drive/MyDrive/PINNs_piezo_20k_outp

In [6]:
campaign(['indirect_arch_3x100'])
# campaign(['indirect_tanh_3x50'])
# campaign(['indirect_routing_off'])

Ejecutando: /usr/bin/python3 -u -m scripts.run_paper20k_campaign --epochs 20000 --device cuda --dtype float64 --experiments indirect_arch_3x100

[START] indirect_arch_3x100 (architecture)
/usr/bin/python3 -u -m scripts.train_indirect --epochs-adam 20000 --device cuda --dtype float64 --epochs-lbfgs 0 --lr-adam 1e-3 --model-type pyramid --hidden-sizes 100 100 100 --activation silu --phase-enriched --no-split-trunks --no-hard-natural-bcs --interior-per-layer 2048 --boundary-points 256 --interface-points 256 --resample-every 200 --interface-weight 1.0 --no-slender-transverse-scaling --normal-constitutive-weight 1.0 --constitutive-normalization represented --no-constitutive-traction-bcs --no-constitutive-flux-stopgrad --mechanical-constitutive-stopgrad --stress-warmup-epochs 0 --seed 20260728 --voltage 100 --no-zero-output --no-periodic-checkpoints --run-name paper20k_indirect_arch_3x100
Using device: cuda
Training dtype: torch.float64
Run directory: /content/drive/MyDrive/PINNs_piezo_20k_o

In [ ]:
campaign(['indirect_tanh_3x50'])

Ejecutando: /usr/bin/python3 -u -m scripts.run_paper20k_campaign --epochs 20000 --device cuda --dtype float64 --experiments indirect_tanh_3x50

[START] indirect_tanh_3x50 (activation)
/usr/bin/python3 -u -m scripts.train_indirect --epochs-adam 20000 --device cuda --dtype float64 --epochs-lbfgs 0 --lr-adam 1e-3 --model-type pyramid --hidden-sizes 50 50 50 --activation tanh --phase-enriched --no-split-trunks --no-hard-natural-bcs --interior-per-layer 2048 --boundary-points 256 --interface-points 256 --resample-every 200 --interface-weight 1.0 --no-slender-transverse-scaling --normal-constitutive-weight 1.0 --constitutive-normalization represented --no-constitutive-traction-bcs --no-constitutive-flux-stopgrad --mechanical-constitutive-stopgrad --stress-warmup-epochs 0 --seed 20260728 --voltage 100 --no-zero-output --no-periodic-checkpoints --run-name paper20k_indirect_tanh_3x50


In [ ]:
campaign(['indirect_routing_off'])

## 4. Sensibilidad de puntos interiores (converso)

El baseline de 2048 puntos ya se obtiene en `indirect_mixed`; aquí se entrenan 512 y 1024.

In [7]:
campaign(['indirect_interior_512'])


Ejecutando: /usr/bin/python3 -u -m scripts.run_paper20k_campaign --epochs 20000 --device cuda --dtype float64 --experiments indirect_interior_512

[START] indirect_interior_512 (collocation_sensitivity)
/usr/bin/python3 -u -m scripts.train_indirect --epochs-adam 20000 --device cuda --dtype float64 --epochs-lbfgs 0 --lr-adam 1e-3 --model-type pyramid --hidden-sizes 50 50 50 --activation silu --phase-enriched --no-split-trunks --no-hard-natural-bcs --interior-per-layer 512 --boundary-points 256 --interface-points 256 --resample-every 200 --interface-weight 1.0 --no-slender-transverse-scaling --normal-constitutive-weight 1.0 --constitutive-normalization represented --no-constitutive-traction-bcs --no-constitutive-flux-stopgrad --mechanical-constitutive-stopgrad --stress-warmup-epochs 0 --seed 20260728 --voltage 100 --no-zero-output --no-periodic-checkpoints --run-name paper20k_indirect_interior_512
Using device: cuda
Training dtype: torch.float64
Run directory: /content/drive/MyDrive/PINN

In [8]:
campaign(['indirect_interior_1024'])

Ejecutando: /usr/bin/python3 -u -m scripts.run_paper20k_campaign --epochs 20000 --device cuda --dtype float64 --experiments indirect_interior_1024

[START] indirect_interior_1024 (collocation_sensitivity)
/usr/bin/python3 -u -m scripts.train_indirect --epochs-adam 20000 --device cuda --dtype float64 --epochs-lbfgs 0 --lr-adam 1e-3 --model-type pyramid --hidden-sizes 50 50 50 --activation silu --phase-enriched --no-split-trunks --no-hard-natural-bcs --interior-per-layer 1024 --boundary-points 256 --interface-points 256 --resample-every 200 --interface-weight 1.0 --no-slender-transverse-scaling --normal-constitutive-weight 1.0 --constitutive-normalization represented --no-constitutive-traction-bcs --no-constitutive-flux-stopgrad --mechanical-constitutive-stopgrad --stress-warmup-epochs 0 --seed 20260728 --voltage 100 --no-zero-output --no-periodic-checkpoints --run-name paper20k_indirect_interior_1024
Using device: cuda
Training dtype: torch.float64
Run directory: /content/drive/MyDrive/

## 5. Barrido de fuerza

El baseline directo aporta 0.10 N; se añaden entrenamientos independientes a 0.05 y 0.20 N.

In [3]:
campaign(['direct_force_0p05N'])
# campaign(['direct_force_0p2N'])


Ejecutando: /usr/bin/python3 -u -m scripts.run_paper20k_campaign --epochs 20000 --device cuda --dtype float64 --experiments direct_force_0p05N

[START] direct_force_0p05N (force_sweep)
/usr/bin/python3 -u -m scripts.train_direct --epochs-adam 20000 --device cuda --dtype float64 --epochs-lbfgs 0 --lr-adam 1e-3 --activation silu --hidden-sizes 50 50 50 --output-init-gain 1.0 --random-collocation --interior-per-layer 2048 --boundary-points 256 --interface-points 256 --resample-every 200 --force -0.05 --traction-profile uniform --electrical-bc floating_electrode --hard-natural-bcs --hard-floating-electrode --slender-warping --phase-enriched --no-split-trunks --pde-weight 1.0 --bc-weight 10.0 --interface-weight 1.0 --no-constitutive-traction-bcs --normal-constitutive-weight 1.0 --strict-transverse-constitutive --constitutive-normalization dominant --lbfgs-grad-clip 0.0 --seed 20260728 --no-periodic-checkpoints --run-name paper20k_direct_force_0p05N
Using device: cuda
Training dtype: torch.f

In [4]:
campaign(['direct_force_0p2N'])


Ejecutando: /usr/bin/python3 -u -m scripts.run_paper20k_campaign --epochs 20000 --device cuda --dtype float64 --experiments direct_force_0p2N

[START] direct_force_0p2N (force_sweep)
/usr/bin/python3 -u -m scripts.train_direct --epochs-adam 20000 --device cuda --dtype float64 --epochs-lbfgs 0 --lr-adam 1e-3 --activation silu --hidden-sizes 50 50 50 --output-init-gain 1.0 --random-collocation --interior-per-layer 2048 --boundary-points 256 --interface-points 256 --resample-every 200 --force -0.2 --traction-profile uniform --electrical-bc floating_electrode --hard-natural-bcs --hard-floating-electrode --slender-warping --phase-enriched --no-split-trunks --pde-weight 1.0 --bc-weight 10.0 --interface-weight 1.0 --no-constitutive-traction-bcs --normal-constitutive-weight 1.0 --strict-transverse-constitutive --constitutive-normalization dominant --lbfgs-grad-clip 0.0 --seed 20260728 --no-periodic-checkpoints --run-name paper20k_direct_force_0p2N
Using device: cuda
Training dtype: torch.float

## 6. Barrido de voltaje

El baseline converso aporta 100 V; se añaden 200, 300, 400 y 500 V, todos con entrenamiento independiente desde cero.

In [3]:
campaign(['indirect_voltage_200V'])
# campaign(['indirect_voltage_300V'])
# campaign(['indirect_voltage_400V'])
# campaign(['indirect_voltage_500V'])


Ejecutando: /usr/bin/python3 -u -m scripts.run_paper20k_campaign --epochs 20000 --device cuda --dtype float64 --experiments indirect_voltage_200V

[START] indirect_voltage_200V (voltage_sweep)
/usr/bin/python3 -u -m scripts.train_indirect --epochs-adam 20000 --device cuda --dtype float64 --epochs-lbfgs 0 --lr-adam 1e-3 --model-type pyramid --hidden-sizes 50 50 50 --activation silu --phase-enriched --no-split-trunks --no-hard-natural-bcs --interior-per-layer 2048 --boundary-points 256 --interface-points 256 --resample-every 200 --interface-weight 1.0 --no-slender-transverse-scaling --normal-constitutive-weight 1.0 --constitutive-normalization represented --no-constitutive-traction-bcs --no-constitutive-flux-stopgrad --mechanical-constitutive-stopgrad --stress-warmup-epochs 0 --seed 20260728 --voltage 200 --no-zero-output --no-periodic-checkpoints --run-name paper20k_indirect_voltage_200V
Using device: cuda
Training dtype: torch.float64
Run directory: /content/drive/MyDrive/PINNs_piezo_2

In [4]:
campaign(['indirect_voltage_300V'])
# campaign(['indirect_voltage_400V'])
# campaign(['indirect_voltage_500V'])


Ejecutando: /usr/bin/python3 -u -m scripts.run_paper20k_campaign --epochs 20000 --device cuda --dtype float64 --experiments indirect_voltage_300V

[START] indirect_voltage_300V (voltage_sweep)
/usr/bin/python3 -u -m scripts.train_indirect --epochs-adam 20000 --device cuda --dtype float64 --epochs-lbfgs 0 --lr-adam 1e-3 --model-type pyramid --hidden-sizes 50 50 50 --activation silu --phase-enriched --no-split-trunks --no-hard-natural-bcs --interior-per-layer 2048 --boundary-points 256 --interface-points 256 --resample-every 200 --interface-weight 1.0 --no-slender-transverse-scaling --normal-constitutive-weight 1.0 --constitutive-normalization represented --no-constitutive-traction-bcs --no-constitutive-flux-stopgrad --mechanical-constitutive-stopgrad --stress-warmup-epochs 0 --seed 20260728 --voltage 300 --no-zero-output --no-periodic-checkpoints --run-name paper20k_indirect_voltage_300V
Using device: cuda
Training dtype: torch.float64
Run directory: /content/drive/MyDrive/PINNs_piezo_2

In [5]:
campaign(['indirect_voltage_400V'])
# campaign(['indirect_voltage_500V'])

Ejecutando: /usr/bin/python3 -u -m scripts.run_paper20k_campaign --epochs 20000 --device cuda --dtype float64 --experiments indirect_voltage_400V

[START] indirect_voltage_400V (voltage_sweep)
/usr/bin/python3 -u -m scripts.train_indirect --epochs-adam 20000 --device cuda --dtype float64 --epochs-lbfgs 0 --lr-adam 1e-3 --model-type pyramid --hidden-sizes 50 50 50 --activation silu --phase-enriched --no-split-trunks --no-hard-natural-bcs --interior-per-layer 2048 --boundary-points 256 --interface-points 256 --resample-every 200 --interface-weight 1.0 --no-slender-transverse-scaling --normal-constitutive-weight 1.0 --constitutive-normalization represented --no-constitutive-traction-bcs --no-constitutive-flux-stopgrad --mechanical-constitutive-stopgrad --stress-warmup-epochs 0 --seed 20260728 --voltage 400 --no-zero-output --no-periodic-checkpoints --run-name paper20k_indirect_voltage_400V
Using device: cuda
Training dtype: torch.float64
Run directory: /content/drive/MyDrive/PINNs_piezo_2

In [3]:
campaign(['indirect_voltage_500V'])

Ejecutando: /usr/bin/python3 -u -m scripts.run_paper20k_campaign --epochs 20000 --device cuda --dtype float64 --experiments indirect_voltage_500V

[START] indirect_voltage_500V (voltage_sweep)
/usr/bin/python3 -u -m scripts.train_indirect --epochs-adam 20000 --device cuda --dtype float64 --epochs-lbfgs 0 --lr-adam 1e-3 --model-type pyramid --hidden-sizes 50 50 50 --activation silu --phase-enriched --no-split-trunks --no-hard-natural-bcs --interior-per-layer 2048 --boundary-points 256 --interface-points 256 --resample-every 200 --interface-weight 1.0 --no-slender-transverse-scaling --normal-constitutive-weight 1.0 --constitutive-normalization represented --no-constitutive-traction-bcs --no-constitutive-flux-stopgrad --mechanical-constitutive-stopgrad --stress-warmup-epochs 0 --seed 20260728 --voltage 500 --no-zero-output --no-periodic-checkpoints --run-name paper20k_indirect_voltage_500V
Using device: cuda
Training dtype: torch.float64
Run directory: /content/drive/MyDrive/PINNs_piezo_2

## Estado de la campaña

In [11]:
from scripts.run_paper20k_campaign import EXPERIMENTS, model_path
import pandas as pd
status = pd.DataFrame([
    {'experiment': name, 'group': exp.group,
     'complete': model_path(exp).exists(),
     'run': exp.run_name}
    for name, exp in EXPERIMENTS.items()
])
display(status)
print(f"Completos: {status['complete'].sum()}/{len(status)}")


,experiment,group,complete,run
0,direct_mixed,baseline,True,paper20k_direct_mixed
1,indirect_mixed,baseline,True,paper20k_indirect_mixed
2,direct_three_field,mixed_vs_three_field,True,paper20k_direct_three_field
3,indirect_three_field,mixed_vs_three_field,True,paper20k_indirect_three_field
4,indirect_arch_2x50,architecture,True,paper20k_indirect_arch_2x50
5,indirect_arch_4x50,architecture,True,paper20k_indirect_arch_4x50
6,indirect_arch_3x100,architecture,True,paper20k_indirect_arch_3x100
7,indirect_tanh_3x50,activation,True,paper20k_indirect_tanh_3x50
8,indirect_routing_off,gradient_routing,True,paper20k_indirect_routing_off
9,indirect_interior_512,collocation_sensitivity,True,paper20k_indirect_interior_512


Completos: 17/17


## Evaluación común y paquete actual

Puedes ejecutar esta celda en cualquier momento. Evalúa todas las corridas terminadas sobre FEM P2 201×21 y genera inmediatamente sus CSV/JSON y figuras PNG+PDF. Vuelve a ejecutarla conforme terminen más corridas; al llegar a 17/17 genera automáticamente el paquete final completo.

In [8]:
stream([sys.executable, '-u', '-m', 'scripts.build_paper20k_campaign', '--allow-partial'])
print('Resultados:', OUTPUTS / 'paper20k_results_package')
print('Métricas:', OUTPUTS / 'runs/paper20k_summary/all_metrics.json')
print('Tabla completa:', OUTPUTS / 'runs/paper20k_summary/ablation_results.csv')


Ejecutando: /usr/bin/python3 -u -m scripts.build_paper20k_campaign --allow-partial
Metrics: /content/drive/MyDrive/PINNs_piezo_20k_outputs/runs/paper20k_summary/all_metrics.json
Table: /content/drive/MyDrive/PINNs_piezo_20k_outputs/runs/paper20k_summary/ablation_results.csv
/content/drive/MyDrive/PINNs_piezo_20k_outputs/paper20k_results_package
Publication package: /content/drive/MyDrive/PINNs_piezo_20k_outputs/paper20k_results_package
Resultados: /content/drive/MyDrive/PINNs_piezo_20k_outputs/paper20k_results_package
Métricas: /content/drive/MyDrive/PINNs_piezo_20k_outputs/runs/paper20k_summary/all_metrics.json
Tabla completa: /content/drive/MyDrive/PINNs_piezo_20k_outputs/runs/paper20k_summary/ablation_results.csv
